# Step 5 — Register in RHOAI Model Registry and deploy with KServe

Three things happen here:
1. Upload the `best.pt` weights from notebook 04 to S3.
2. Register a new `ModelVersion` in the RHOAI Model Registry pointing at the S3 URI.
3. Create a KServe `InferenceService` that serves those weights.

**Cluster prerequisites** (the workshop lab environment provides these):
- An S3-compatible object store reachable from the workbench (ODF / MinIO / RGW).
- The RHOAI Model Registry component enabled (`kubectl get modelregistry -A`).
- A serving runtime on the cluster that can load an Ultralytics `.pt` file. This notebook uses a custom `ServingRuntime` shipped by the lab (`ultralytics-yolo-runtime`). Swap to your own runtime name if it differs.

In [1]:
%%capture
!pip install boto3==1.35.55 botocore==1.35.55 model-registry kubernetes requests pyyaml

In [2]:
import os
import time
from pathlib import Path

import boto3
import botocore

## 5a. Upload `best.pt` to S3

Uses the standard RHOAI data-connection env vars injected into the workbench. The model lands at `s3://$BUCKET_NAME/models/ppe-yolov8n-vlm/v1/best.pt`.

In [5]:
aws_access_key_id     = os.environ["AWS_ACCESS_KEY_ID"]
aws_secret_access_key = os.environ["AWS_SECRET_ACCESS_KEY"]
endpoint_url          = os.environ["AWS_S3_ENDPOINT"]
region_name           = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
bucket_name           = os.environ.get("AWS_S3_BUCKET", "storage")

MODEL_NAME    = "ppe-yolov8n-vlm"
MODEL_VERSION = "1"
S3_PREFIX     = f"models/{MODEL_NAME}/{MODEL_VERSION}"

# Ultralytics may write to `runs/detect/<name>/...` or a nested path depending on
# its settings. Find the newest best.pt anywhere under runs/ instead of hardcoding.
candidates = sorted(Path("runs").rglob("best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, "No best.pt found under runs/ — run notebook 04 first."
LOCAL_WEIGHTS = candidates[0]
print(f"Using trained weights: {LOCAL_WEIGHTS}")

session = boto3.session.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
)
s3 = session.resource(
    "s3",
    config=botocore.client.Config(signature_version="s3v4"),
    endpoint_url=endpoint_url,
    region_name=region_name,
)
bucket = s3.Bucket(bucket_name)

Using trained weights: runs/detect/ppe-vlm/weights/best.pt


In [6]:
s3_key = f"{S3_PREFIX}/best.pt"
bucket.upload_file(str(LOCAL_WEIGHTS), s3_key)

# Also upload the class names so the serving runtime can label predictions.
bucket.upload_file("data-vlm.yaml", f"{S3_PREFIX}/data.yaml")

s3_uri = f"s3://{bucket_name}/{S3_PREFIX}"
print("Uploaded weights to:", s3_uri)

Uploaded weights to: s3://models/models/ppe-yolov8n-vlm/1


## 5b. Register the model in RHOAI Model Registry

`MODEL_REGISTRY_URL` is the route of the Model Registry service on your cluster — the lab will inject it as an env var. Get it manually with:

```bash
oc get route -n rhoai-model-registries modelregistry-sample -o jsonpath='{.spec.host}'
```

In [8]:
import csv
import datetime

import yaml
from model_registry import ModelRegistry

# --- Harvest training hyperparameters and final metrics from the Ultralytics run ---
# Both files sit beside best.pt. We made LOCAL_WEIGHTS point at it earlier.
run_dir = LOCAL_WEIGHTS.parent.parent  # weights/best.pt -> weights/ -> run root

args_path    = run_dir / "args.yaml"
results_path = run_dir / "results.csv"

train_args = yaml.safe_load(args_path.read_text()) if args_path.is_file() else {}
final_row  = {}
if results_path.is_file():
    rows = list(csv.DictReader(results_path.open()))
    if rows:
        final_row = rows[-1]

def _fmt(v, digits=4):
    try:
        return f"{float(v):.{digits}f}"
    except (TypeError, ValueError):
        return "" if v is None else str(v)

# --- Build the metadata payload ---
# The RHOAI Model Registry stores these as `custom_properties` on the ModelVersion.
# We namespace keys with `tag.` / `param.` / `metric.` prefixes so the dashboard
# groups them visually and the provenance is self-documenting.
metadata = {
    # --- Tags (descriptive labels) ---
    "tag.workshop":       "rhelai-cv-ppe-demo",
    "tag.task":           "object_detection",
    "tag.framework":      "ultralytics",
    "tag.architecture":   "yolov8n",
    "tag.base_model":     "yolov8n.pt (COCO-pretrained)",
    "tag.annotator":      "Qwen2.5-VL-7B-Instruct",
    "tag.source_dataset": "construction-ppe (VLM-annotated subset)",
    "tag.num_classes":    "7",
    "tag.classes":        "person,helmet,no-helmet,vest,no-vest,gloves,no-gloves",
    "tag.registered_at":  datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),

    # --- Hyperparameters used for training ---
    "param.epochs":    _fmt(train_args.get("epochs"), digits=0),
    "param.imgsz":     _fmt(train_args.get("imgsz"), digits=0),
    "param.batch":     _fmt(train_args.get("batch"), digits=0),
    "param.lr0":       _fmt(train_args.get("lr0"), digits=6),
    "param.optimizer": str(train_args.get("optimizer", "")),
    "param.freeze":    _fmt(train_args.get("freeze"), digits=0),
    "param.patience":  _fmt(train_args.get("patience"), digits=0),

    # --- Final-epoch metrics (whole-dataset, all classes) ---
    "metric.final_epoch": _fmt(final_row.get("epoch"), digits=0),
    "metric.mAP50":       _fmt(final_row.get("metrics/mAP50(B)")),
    "metric.mAP50_95":    _fmt(final_row.get("metrics/mAP50-95(B)")),
    "metric.precision":   _fmt(final_row.get("metrics/precision(B)")),
    "metric.recall":      _fmt(final_row.get("metrics/recall(B)")),
}

# --- Register with the RHOAI Model Registry ---
MODEL_REGISTRY_URL = "https://user1-registry-rest.apps.ocp.9xgvv.sandbox3434.opentlc.com:443/api/model_registry/v1alpha3"
registry = ModelRegistry(
    server_address=MODEL_REGISTRY_URL,
    port=int(os.environ.get("MODEL_REGISTRY_PORT", 443)),
    author=os.environ.get("JUPYTER_USER", "workshop"),
    user_token=os.environ.get("MODEL_REGISTRY_TOKEN"),
    is_secure=True,
)

description = (
    "YOLOv8n fine-tuned on Qwen2.5-VL zero-shot annotations. "
    "7 PPE-compliance classes. "
    f"mAP50={metadata['metric.mAP50'] or '?'} "
    f"after {metadata['param.epochs'] or '?'} epochs "
    f"(fine-tuning head only, backbone frozen)."
)

rm = registry.register_model(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    model_format_name="pytorch",
    model_format_version="1",
    model_source_kind="s3",
    model_source_class="ultralytics",
    model_source_group="yolov8",
    model_source_id=MODEL_NAME,
    model_source_name="best.pt",
    uri=s3_uri,
    description=description,
    metadata=metadata,
)

# --- Audit printout: what the workshop audience will see in the RHOAI dashboard ---
print("Registered:", rm)
print("\nModel Registry custom properties:")
for k in sorted(metadata):
    print(f"  {k:24s} = {metadata[k]}")

ServiceException: (503)
Reason: Service Unavailable
HTTP response headers: <CIMultiDictProxy('Pragma': 'no-cache', 'Cache-Control': 'private, max-age=0, no-cache, no-store', 'Content-Type': 'text/html')>
HTTP response body: <html>
  <head>
    <meta name="viewport" content="width=device-width, initial-scale=1">

    <style type="text/css">
      body {
        font-family: "Helvetica Neue", Helvetica, Arial, sans-serif;
        line-height: 1.66666667;
        font-size: 16px;
        color: #333;
        background-color: #fff;
        margin: 2em 1em;
      }
      h1 {
        font-size: 28px;
        font-weight: 400;
      }
      p {
        margin: 0 0 10px;
      }
      .alert.alert-info {
        background-color: #F0F0F0;
        margin-top: 30px;
        padding: 30px;
      }
      .alert p {
        padding-left: 35px;
      }
      ul {
        padding-left: 51px;
        position: relative;
      }
      li {
        font-size: 14px;
  

## 5c. Deploy with KServe

We create an `InferenceService` that pulls the weights from S3. The workbench's S3 credentials are referenced via a pre-existing `Secret` named `aws-connection-storage` (created by the RHOAI Data Connection) — KServe reads it via the `serviceAccountName: aws-connection-storage`.

In [ ]:
NAMESPACE        = os.environ.get("KUBERNETES_NAMESPACE", "ppe-detection-cv")
SERVING_RUNTIME  = os.environ.get("SERVING_RUNTIME", "ultralytics-yolo-runtime")
SERVICE_ACCOUNT  = os.environ.get("SERVICE_ACCOUNT", "aws-connection-storage")

isvc_manifest = {
    "apiVersion": "serving.kserve.io/v1beta1",
    "kind": "InferenceService",
    "metadata": {
        "name": MODEL_NAME,
        "namespace": NAMESPACE,
        "labels": {"opendatahub.io/dashboard": "true"},
        "annotations": {
            "serving.knative.openshift.io/enablePassthrough": "true",
            "sidecar.istio.io/inject": "true",
            "sidecar.istio.io/rewriteAppHTTPProbers": "true",
        },
    },
    "spec": {
        "predictor": {
            "serviceAccountName": SERVICE_ACCOUNT,
            "model": {
                "runtime": SERVING_RUNTIME,
                "modelFormat": {"name": "pytorch"},
                "storageUri": s3_uri,
                "resources": {
                    "requests": {"cpu": "500m", "memory": "1Gi"},
                    "limits":   {"cpu": "2",   "memory": "4Gi"},
                },
            },
        }
    },
}

print(isvc_manifest)

In [ ]:
from kubernetes import client, config

try:
    config.load_incluster_config()
except config.ConfigException:
    config.load_kube_config()

api = client.CustomObjectsApi()
group, version, plural = "serving.kserve.io", "v1beta1", "inferenceservices"

try:
    api.create_namespaced_custom_object(group, version, NAMESPACE, plural, isvc_manifest)
    print("Created InferenceService", MODEL_NAME)
except client.ApiException as e:
    if e.status == 409:
        api.patch_namespaced_custom_object(group, version, NAMESPACE, plural, MODEL_NAME, isvc_manifest)
        print("Patched existing InferenceService", MODEL_NAME)
    else:
        raise

## Wait until the InferenceService is Ready, then print the route

In [ ]:
TIMEOUT_S = 600
deadline  = time.time() + TIMEOUT_S
url = None

while time.time() < deadline:
    isvc = api.get_namespaced_custom_object(group, version, NAMESPACE, plural, MODEL_NAME)
    status = isvc.get("status", {})
    ready = next(
        (c for c in status.get("conditions", []) if c.get("type") == "Ready"),
        None,
    )
    if ready and ready.get("status") == "True":
        url = status.get("url") or status.get("address", {}).get("url")
        break
    print(f"  … waiting, Ready={ready.get('status') if ready else 'unknown'}")
    time.sleep(10)

if not url:
    raise TimeoutError(f"InferenceService {MODEL_NAME} did not reach Ready=True in {TIMEOUT_S}s.")

print("\nEndpoint:", url)

# Persist for the next notebook.
Path("output").mkdir(exist_ok=True)
Path("output/inference_endpoint.txt").write_text(url)

---

**Next:** [06-consume-deployed-endpoint.ipynb](06-consume-deployed-endpoint.ipynb) — POST an image at the endpoint above and draw the predicted boxes.